# Building a Sales Proposal with a Heterogeneous Agent Team

We'll use Claude Managed Agents and the callable_agents feature to build a sales proposal for a fictional product called Northstar, a workflow-automation platform for mid-market operations teams.

Right now, their reps build a tailored proposal for each prospect: research what companies in the prospect's segment typically prioritize, pull two relevant case studies from a library of a few hundred, model pricing from an internal rules sheet, and assemble it into a two-page document. Each step draws on a different source and a different kind of judgment.

We'll have a coordinator agent run three specialists to do this. A researcher gets web search and finds the prospect's priorities. A librarian reads the case-study library and picks the two best matches. A pricing modeler sees only the rules file and the seat count. The coordinator sequences them and writes the proposal.

## 1. Set up the client

First, let's install the SDK and set up the Anthropic client. The `callable_agents` field and the multiagent event types are currently behind the research-preview header.


In [ ]:
%pip install anthropic

In [ ]:
import os, time, httpx, anthropic

PREVIEW = "managed-agents-2026-04-01-research-preview"
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-opus-4-7")
client = anthropic.Anthropic()


# The SDK does not yet type the research-preview event types (thread_created,
# thread_message_received, ...), so we read events as raw JSON.
def list_events(session_id):
    r = httpx.get(
        f"https://api.anthropic.com/v1/sessions/{session_id}/events",
        params={"limit": 1000},
        headers={
            "x-api-key": client.api_key,
            "anthropic-version": "2023-06-01",
            "anthropic-beta": f"managed-agents-2026-04-01,{PREVIEW}",
        },
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["data"]

## 2. Define three specialist subagents

Next, we'll create the three teammates. Each one gets its own system prompt, its own output schema, and only the tools it needs for its job. The researcher gets web search, the case-study picker can only read the local library, and the pricing modeler just sees `pricing_rules.md` and the seat count. Scoping tools per role keeps the pricer from pulling a competitor's number off the web and keeps the full case-study library out of the coordinator's context.


In [ ]:
def make_agent(name, description, system, tools):
    a = client.beta.agents.create(
        name=name,
        description=description,
        model=MODEL,
        system=system,
        tools=tools,
        betas=[PREVIEW],
    )
    print(f"{name}: {a.id}")
    return {"type": "agent", "id": a.id}


prospect_researcher = make_agent(
    "prospect_researcher",
    "Researches what companies in a given industry segment and size tier typically prioritize.",
    """Given a prospect's industry and size, use web search to find:
- What companies in that segment typically list as strategic priorities
- Recent trends or pressures in that industry
- Common operational pain points at that scale
Return via send_to_parent: {"priorities": [...], "recent_moves": [...], "pain_points": [...], "sources": [...]}""",
    [
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
)

case_study_picker = make_agent(
    "case_study_picker",
    "Selects the two most relevant case studies from the library for a given prospect profile.",
    """The case study library is in /mnt/user-data/case_studies/. Each file is one customer story.
You will be given a prospect's industry, size, and top priorities. Read the library, score each study on relevance, and pick the two best matches.
Return via send_to_parent: {"picks": [{"file": ..., "customer": ..., "why_relevant": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

pricing_modeler = make_agent(
    "pricing_modeler",
    "Builds two or three pricing options for a prospect based on seat count and expected usage.",
    """Pricing rules are in /mnt/user-data/pricing_rules.md. Given a prospect's estimated seat count and usage tier, build:
- a conservative option (annual commit, lower per-seat)
- a flexible option (monthly, higher per-seat)
- if seat count > 500, an enterprise option with a platform fee
Show the first-year total for each. Return via send_to_parent: {"options": [{"name": ..., "structure": ..., "year_one_total": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

## 3. Give the team something to work with

The librarian needs a library to choose from. We'll give it seven short case studies across healthcare, manufacturing, logistics, retail, fintech, and public sector, so you can see it actually pick the two that fit our prospect.


In [10]:
CASE_STUDIES = [
    {
        "slug": "stclair_health",
        "title": "St. Clair Health",
        "industry": "regional hospital network",
        "employees": 6200,
        "summary": """Challenge: credentialing and prior-auth workflows spread across 11 systems.
Result with Northstar: consolidated to 3 automated workflows; prior-auth turnaround down 58%; $1.9M annual labor savings.""",
    },
    {
        "slug": "blueridge_health_plan",
        "title": "BlueRidge Health Plan",
        "industry": "regional payer",
        "employees": 2800,
        "summary": """Challenge: claims-adjudication exceptions queued in email; 19% required manual rework.
Result with Northstar: exception routing automated end-to-end; rework rate down to 6%; 11-day faster average claim resolution.""",
    },
    {
        "slug": "calder_mfg",
        "title": "Calder Manufacturing",
        "industry": "industrial",
        "employees": 3100,
        "summary": """Challenge: purchase-order approvals averaging 9 days.
Result with Northstar: PO cycle time cut to 2.1 days; 14% reduction in maverick spend.""",
    },
    {
        "slug": "northwind",
        "title": "Northwind Logistics",
        "industry": "3PL",
        "employees": 4400,
        "summary": """Challenge: carrier-onboarding paperwork took 3 weeks per carrier.
Result with Northstar: onboarding down to 4 days; 22% more carriers activated in Q1.""",
    },
    {
        "slug": "harborview_retail",
        "title": "Harborview Retail Group",
        "industry": "specialty retail",
        "employees": 5600,
        "summary": """Challenge: store-level inventory exceptions handled by regional managers over Slack and spreadsheets.
Result with Northstar: exception triage automated across 140 stores; stockout incidents down 31%.""",
    },
    {
        "slug": "aperture_fintech",
        "title": "Aperture Payments",
        "industry": "fintech",
        "employees": 1900,
        "summary": """Challenge: KYC and merchant-onboarding reviews averaging 6 business days.
Result with Northstar: review SLA cut to 36 hours; onboarding throughput up 2.4x with the same team.""",
    },
    {
        "slug": "summit_county",
        "title": "Summit County Government",
        "industry": "public sector",
        "employees": 3700,
        "summary": """Challenge: building-permit applications routed through five departments by paper packet.
Result with Northstar: single digital intake with parallel department review; median permit time 41 to 17 days.""",
    },
]

### Product and pricing collateral

We'll also provide the product one-pager that the coordinator reads when writing the "How we help" section, and the pricing rules file that the modeler uses to build options.


In [11]:
PRODUCT = """# Northstar Platform — One-Pager
Northstar is a workflow automation platform for mid-market operations teams.
Core capabilities: visual process builder, 200+ SaaS connectors, role-based approvals, SOC 2 Type II.
Typical results: 40-60% reduction in manual ticket handling, 3-week time-to-first-workflow."""

PRICING = """# Pricing Rules (internal)
- Per-seat list: $65/mo (monthly) or $52/mo (annual commit).
- Usage tiers: light = 1.0x, standard = 1.15x, heavy = 1.30x multiplier on per-seat.
- Enterprise (>500 seats): add $48,000/yr platform fee, per-seat drops to $44/mo annual.
- All options include onboarding; enterprise includes a named CSM."""

### Wire up the coordinator and start a session

Now let's create an environment, upload the nine files, and create the coordinator with its three `callable_agents`. Each entry is a full agent with its own model, prompt, and toolset, so you could mix model tiers per role.


In [ ]:
env = client.beta.environments.create(name="proposal-meridian", betas=[PREVIEW])

resources = []


def mount(path, content):
    f = client.beta.files.upload(file=(os.path.basename(path), content.encode(), "text/plain"))
    resources.append({"type": "file", "file_id": f.id, "mount_path": path})


for cs in CASE_STUDIES:
    body = f"# {cs['title']} ({cs['industry']}, {cs['employees']:,} employees)\n{cs['summary']}"
    mount(f"/mnt/user-data/case_studies/{cs['slug']}.md", body)
mount("/mnt/user-data/product_one_pager.md", PRODUCT)
mount("/mnt/user-data/pricing_rules.md", PRICING)

coordinator = client.beta.agents.create(
    name="Proposal Writer",
    model=MODEL,
    system="""You assemble tailored sales proposals.
Given a prospect name and basic profile:
1. Send the prospect's industry and size to prospect_researcher.
2. Send the prospect's industry, size, and (once the researcher reports back) their priorities to case_study_picker.
3. Send the seat count and usage tier to pricing_modeler.
4. Read /mnt/user-data/product_one_pager.md, then write /mnt/session/outputs/proposal.md with sections:
   Executive summary (tied to their priorities), How we help (from the one-pager),
   Proof (the two case studies), Investment (the pricing options), Next steps.
Keep it to two pages.""",
    tools=[{"type": "agent_toolset_20260401"}],
    betas=[PREVIEW],
    extra_body={"callable_agents": [prospect_researcher, case_study_picker, pricing_modeler]},
)

session = client.beta.sessions.create(
    agent=coordinator.id,
    environment_id=env.id,
    resources=resources,
    title="Proposal: Meridian Health",
    betas=[PREVIEW],
)
print(f"Session {session.id} ready with {len(resources)} files mounted")

## 4. Kick off the proposal

Let's send the prospect profile and watch the coordinator work. It will start the researcher and the pricing modeler in parallel, then run the case-study picker once the researcher's findings come back, since the picker needs those priorities to score relevance.


In [14]:
PROSPECT = {
    "name": "Meridian Health",
    "industry": "regional healthcare system",
    "employees": 8500,
    "estimated_seats": 600,
    "usage_tier": "heavy",
}

client.beta.sessions.events.send(
    session.id,
    betas=[PREVIEW],
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": f"Build a proposal for {PROSPECT['name']}, a {PROSPECT['industry']} with "
                    f"~{PROSPECT['employees']} employees. Estimate {PROSPECT['estimated_seats']} seats "
                    f"at {PROSPECT['usage_tier']} usage. Write to /mnt/session/outputs/proposal.md.",
                }
            ],
        }
    ],
)

seen, created, idled = set(), 0, 0
while True:
    for ev in list_events(session.id):
        if ev["id"] in seen:
            continue
        seen.add(ev["id"])
        t = ev["type"]
        if t == "session.thread_created":
            created += 1
            print(f"[spawn] {ev['agent_name']}")
        elif t == "agent.thread_message_received":
            print(f"[report] {ev.get('from_agent_name', 'subagent')} returned")
        elif t == "session.thread_idle":
            idled += 1
        elif t == "session.status_idle":
            # The coordinator idles between dispatch waves; only stop once every
            # spawned thread has finished.
            if created > 0 and idled >= created:
                print(f"[done] {created} subagents finished")
                break
    else:
        time.sleep(4)
        continue
    break

[spawn] prospect_researcher
[spawn] pricing_modeler
[report] prospect_researcher returned
[spawn] case_study_picker
[report] pricing_modeler returned
[report] case_study_picker returned
[done] 3 subagents finished


### What each teammate sent back

Before we look at the assembled proposal, let's print the three raw `send_to_parent` payloads. Each subagent ran in its own context with only its own tools, so the three reports look quite different from one another.


In [15]:
def text_of(content):
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") for b in content if b.get("type") == "text")


for ev in list_events(session.id):
    if ev["type"] == "agent.thread_message_received":
        who = ev["from_agent_name"]
        body = text_of(ev["content"])
        print(f"━━━ send_to_parent from {who} ({len(body)} chars) ━━━")
        print(body[:1200] + (f"\n…[{len(body) - 1200} more chars]" if len(body) > 1200 else ""))
        print()

━━━ send_to_parent from prospect_researcher (12917 chars) ━━━
Research complete for Meridian Health (regional multi-hospital/multi-clinic system, ~8,500 employees). Concise, sales-ready intelligence below.

{
  "priorities": [
    "Financial resilience & margin protection — revenue cycle optimization, cash flow acceleration, and disciplined capital allocation amid reimbursement pressure. <cite index=\"7-2,7-3\">Medicaid changes could create tens of millions in revenue shortfalls for some systems; hospitals must optimize revenue cycle performance, reduce denials and improve margins, while pursuing cost control through efficiency and throughput rather than across-the-board cuts</cite>.",
    "Workforce stabilization & clinician burnout reduction — retention, safer staffing ratios, and reducing administrative burden. <cite index=\"21-3\">Healthcare leaders are focused on five critical priorities to stabilize the nursing profession: staffing and recruitment, competitive pay and benefits, s

## 5. Read the proposal

Finally, let's render the assembled proposal. The coordinator wrote it to `proposal.md` with the `write` tool, so we'll find that event in the log and display it.


In [16]:
from IPython.display import Markdown, display

for ev in list_events(session.id):
    if (
        ev["type"] == "agent.tool_use"
        and ev["name"] == "write"
        and ev["input"]["file_path"].endswith("proposal.md")
    ):
        display(Markdown(ev["input"]["content"]))
        break

# Proposal for Meridian Health
**Northstar Platform — Workflow Automation for a Regional Health System**

---

## Executive Summary

Meridian Health is navigating the same three-front pressure facing every regional IDN in 2026: margin compression from reimbursement and denials, a workforce stretched thin by documentation and administrative burden, and a board-level mandate to scale automation with governance and measurable ROI — all without adding IT headcount or cyber exposure. You are too large to keep running on point solutions, and too lean to build custom integrations for every workflow.

Northstar is built for exactly this middle. We help regional health systems:

- **Protect margin** by automating revenue-cycle workflows (prior-auth, denials, exception handling) that currently leak labor dollars and cash.
- **Give clinician time back** by removing manual handoffs, duplicate data entry, and email-based approvals from credentialing, scheduling, and back-office processes.
- **Consolidate technical debt** by replacing sprawl of point tools with governed, auditable workflows that sit on top of your existing Epic/MEDITECH/Oracle Health environment — no rip-and-replace.
- **Stay audit-ready** on HIPAA and the updated Security Rule with SOC 2 Type II controls, role-based approvals, and full audit trails out of the box.

For a 600-seat heavy-usage deployment, we recommend the **Enterprise tier at $459,840 in year one**, with a named Customer Success Manager to drive adoption across service lines.

---

## How We Help

Northstar is a workflow automation platform for mid-market operations teams. Four capabilities do the heavy lifting at Meridian:

| Capability | What it does for Meridian |
|---|---|
| **Visual process builder** | RCM, credentialing, and ops leads design and modify workflows without engineering tickets — reducing dependence on a stretched IT team. |
| **200+ SaaS connectors** | Bridges existing systems (EHR, payer portals, HRIS, ITSM, finance) via standards-based integration instead of custom point-to-point work. |
| **Role-based approvals** | Enforces governance, segregation of duties, and audit trails required for HIPAA, SOC 2, and internal compliance. |
| **SOC 2 Type II** | BAA-ready, with encryption in transit and at rest, MFA, and vendor-oversight documentation aligned to the updated HIPAA Security Rule. |

**Typical customer results:** 40–60% reduction in manual ticket handling and first production workflow live in 3 weeks — not 3 quarters.

---

## Proof

### St. Clair Health — Regional hospital network, 6,200 employees
**Challenge.** Credentialing and prior-authorization workflows were spread across 11 disconnected systems, creating manual handoffs, delays, and administrative load on clinical and back-office staff.
**What we did.** Consolidated 11 systems into 3 automated, role-based workflows using Northstar's visual builder and connector library — bridging existing systems rather than replacing them.
**Results.**
- **Prior-auth turnaround time down 58%**
- **$1.9M annual labor savings**
- **11 point systems consolidated to 3 governed workflows**

> *Why it matters for Meridian:* the closest analog in our library — same industry, similar size, and a direct hit on your RCM, clinician-burden, and platform-consolidation priorities.

### BlueRidge Health Plan — Regional health payer, 2,800 employees
**Challenge.** Claims-adjudication exceptions were queued informally over email; 19% of claims required manual rework, driving cycle time and labor cost.
**What we did.** Automated end-to-end exception routing with role-based approvals, replacing the email-and-spreadsheet workflow with a governed, auditable process.
**Results.**
- **Manual rework rate cut from 19% to 6%**
- **Average claim resolution 11 days faster**
- **Auditable, HIPAA-aligned exception routing**

> *Why it matters for Meridian:* same adjudication mechanics your RCM team fights from the provider side — evidence Northstar handles HIPAA-sensitive, high-volume claims work with measurable denials and cycle-time impact.

---

## Investment

Northstar Platform sized for **600 seats at the heavy usage tier**. All options include onboarding and SOC 2 Type II controls.

| Option | Commitment | Per-Seat (Heavy) | Platform Fee | **Year-One Total** |
|---|---|---|---|---|
| Flex (Monthly) | Month-to-month | $84.50 / seat / mo | — | **$608,400** |
| Standard (Annual) | 12-month commit | $67.60 / seat / mo | — | **$486,720** |
| **Enterprise** ⭐ | 12-month commit | $57.20 / seat / mo | $48,000 / yr | **$459,840** |

**Recommended: Enterprise** — the lowest year-one total *and* the only tier with a named Customer Success Manager. Saves $26,880 vs. Standard and $148,560 vs. Flex, and gives Meridian a single point of accountability as you roll out across service lines.

At St. Clair Health's results, a comparable deployment would pay for itself roughly **four times over in year one** on labor savings alone — before counting denials recovery, turnover reduction, or cycle-time gains.

---

## Next Steps

1. **Discovery workshop (½ day, on-site or virtual)** — Northstar solutions architect with Meridian's RCM, credentialing, and IT leads to map 3–5 candidate workflows and quantify baseline labor and cycle-time metrics.
2. **Tailored ROI model** — delivered within one week of discovery, tied to Meridian's denials rate, FTE costs, and target workflows.
3. **Security & compliance review** — SOC 2 Type II report, BAA, and HIPAA Security Rule control mapping shared with your CISO's team in parallel.
4. **Pilot-to-production plan** — first workflow live in 3 weeks, with a named CSM driving expansion across service lines under the Enterprise agreement.

**Primary contact:** [Account Executive, Northstar] • Ready to schedule discovery within the week.


## Why three subagents instead of one

A single agent with all three tools could write this proposal, so why split it up? Scoping each role to its own tools means the pricing modeler can't pull a competitor's list price off the web, because it only has the rules file. The case-study picker reads seven files here, but in production it would read hundreds, and that volume stays in the subagent's context instead of the coordinator's. And the coordinator gets to decide the order and the hand-offs without doing any of the specialist work itself.

For more on `callable_agents`, see the [Managed Agents documentation](https://platform.claude.com/docs/en/managed-agents/multi-agent).
